In [1]:
import pandas as pd
import numpy as np
import time
from concurrent.futures import ThreadPoolExecutor

# Simulating a simple database with tables using pandas
def create_sample_data():
    # Create a sample 'users' table
    users = pd.DataFrame({
        'user_id': np.arange(1, 1001),
        'name': [f'User{i}' for i in range(1, 1001)],
        'age': np.random.randint(18, 60, 1000),
        'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston'], 1000)
    })

    # Create a sample 'orders' table
    orders = pd.DataFrame({
        'order_id': np.arange(1, 1001),
        'user_id': np.random.randint(1, 1001, 1000),
        'amount': np.random.randint(10, 500, 1000),
        'date': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(1, 365, 1000), 'D')
    })

    return users, orders

# Simulating an expensive query
def run_query(users, orders, filter_age=None, filter_city=None):
    if filter_age:
        users = users[users['age'] > filter_age]
    if filter_city:
        users = users[users['city'] == filter_city]

    # Join the tables (simulating an expensive query)
    query_result = pd.merge(users, orders, on='user_id', how='inner')

    # Simulate a computation (e.g., sum of order amounts)
    result = query_result.groupby('name')['amount'].sum()

    return result

# Parallel execution of queries
def parallel_query_execution(users, orders):
    # Define multiple queries with different filters for parallel execution
    queries = [
        {'filter_age': 30, 'filter_city': 'New York'},
        {'filter_age': 25, 'filter_city': 'Chicago'},
        {'filter_age': 35, 'filter_city': 'Los Angeles'},
        {'filter_age': 40, 'filter_city': 'Houston'}
    ]

    # Using ThreadPoolExecutor for parallel execution
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(run_query, users, orders, **query) for query in queries]

        # Collect results
        results = [future.result() for future in futures]

    return results

# Main function to execute the queries in parallel
def main():
    # Create sample data
    users, orders = create_sample_data()

    # Start timer to measure execution time
    start_time = time.time()

    # Execute queries in parallel
    results = parallel_query_execution(users, orders)

    # Print results
    for idx, result in enumerate(results):
        print(f"Query {idx+1} Result:")
        print(result.head())  # Displaying top results for brevity
        print("-" * 50)

    # End timer and print execution time
    end_time = time.time()
    print(f"Total execution time: {end_time - start_time} seconds")

# Run the main function
if __name__ == "__main__":
    main()


Query 1 Result:
name
User1000     346
User104     1030
User113      494
User150     1311
User168      456
Name: amount, dtype: int64
--------------------------------------------------
Query 2 Result:
name
User110    296
User128    753
User132    339
User136    537
User138     52
Name: amount, dtype: int64
--------------------------------------------------
Query 3 Result:
name
User102    154
User105    457
User109    197
User148    422
User149    367
Name: amount, dtype: int64
--------------------------------------------------
Query 4 Result:
name
User10     479
User101     87
User107    851
User111    430
User112    367
Name: amount, dtype: int64
--------------------------------------------------
Total execution time: 0.058403968811035156 seconds
